<a href="https://colab.research.google.com/github/royprashant3102-pixel/LA0-TO-ENG-AND-VICEVERSA/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install "transformers>=4.44,<5" "datasets>=2.20" "sentencepiece>=0.2.0" "sacrebleu>=2.4.0" sacremoses accelerate
print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.1 MB/s eta 0:00:00
Dependencies installed.


In [3]:
import os, random, inspect, json, shutil
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
import sentencepiece as spm
import sacrebleu

SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)

# CONFIG (edit knobs here)
CONFIG = {
    "drive_dir": "/content/drive/MyDrive/nmt",
    "max_pairs": 144000,
    "val_size": 2000, "test_size": 2000,
    "base_en2lo": "Helsinki-NLP/opus-mt-en-mul",
    "base_lo2en": "Helsinki-NLP/opus-mt-th-en",
    "tgt_lang_token": ">>tha<<",
    "lao_spm_vocab_size": 8000, "max_new_tokens_to_add": 6000,
    "max_len": 128, "epochs": 3, "batch_size": 16, "grad_accum": 2,
    "lr": 5e-5, "label_smoothing": 0.1, "num_beams": 5,
}
CONFIG["output_root"] = os.path.join(CONFIG["drive_dir"], "runs")
CONFIG["en_file"]     = os.path.join(CONFIG["drive_dir"], "train.en")
CONFIG["lo_file"]     = os.path.join(CONFIG["drive_dir"], "train.lo")

#  DATA
_SAMPLE = [
    ("hello","ສະບາຍດີ"),("thank you","ຂອບໃຈ"),("good morning","ສະບາຍດີຕອນເຊົ້າ"),
    ("how are you","ສະບາຍດີບໍ່"),("i love you","ຂ້ອຍຮັກເຈົ້າ"),("yes","ແມ່ນ"),("no","ບໍ່"),
    ("water","ນ້ຳ"),("food","ອາຫານ"),("school","ໂຮງຮຽນ"),("teacher","ຄູ"),("book","ປຶ້ມ"),
    ("country","ປະເທດ"),("friend","ເພື່ອນ"),("today","ມື້ນີ້"),
]

def _read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [ln.rstrip("\n") for ln in f]

def prepare_data(cfg):
    """Use Drive data if present, else upload ONCE and copy it to Drive."""
    os.makedirs(cfg["drive_dir"], exist_ok=True)
    if os.path.exists(cfg["en_file"]) and os.path.exists(cfg["lo_file"]):
        print("Found data on Drive -> no upload needed.")
        return cfg
    print("No data on Drive yet. Upload once; it will be saved to Drive for next time.")
    from google.colab import files
    print("Upload ENGLISH file:"); up = files.upload(); en_tmp = "/content/" + list(up.keys())[0]
    print("Upload LAO file:");     up = files.upload(); lo_tmp = "/content/" + list(up.keys())[0]
    shutil.copy(en_tmp, cfg["en_file"]); shutil.copy(lo_tmp, cfg["lo_file"])
    print("Saved to Drive:", cfg["en_file"], cfg["lo_file"])
    return cfg

def build_datasets(cfg):
    if os.path.exists(cfg["en_file"]) and os.path.exists(cfg["lo_file"]):
        en, lo = _read_lines(cfg["en_file"]), _read_lines(cfg["lo_file"])
        assert len(en) == len(lo), f"line-count mismatch: {len(en)} vs {len(lo)}"
        print(f"Loaded {len(en):,} raw pairs.")
    else:
        print("No data files - using tiny built-in sample.")
        en = [e for e,_ in _SAMPLE]*8; lo = [l for _,l in _SAMPLE]*8
    pairs = [(e.strip(), l.strip()) for e,l in zip(en,lo) if e.strip() and l.strip()]
    pairs = list(dict.fromkeys(pairs)); random.Random(SEED).shuffle(pairs)
    pairs = pairs[: cfg["max_pairs"]]
    n = len(pairs)
    val_n  = min(cfg["val_size"],  max(1, n//10))
    test_n = min(cfg["test_size"], max(1, n//10))
    val, test = pairs[:val_n], pairs[val_n:val_n+test_n]
    train = pairs[val_n+test_n:] or pairs
    to_ds = lambda p: Dataset.from_dict({"en":[a for a,_ in p], "lo":[b for _,b in p]})
    ds = DatasetDict(train=to_ds(train), validation=to_ds(val), test=to_ds(test))
    print(ds)
    if ds["train"].num_rows < 1000:
        print("*** WARNING: tiny data - check your files. ***")
    return ds

# LAO TOKENS (cached to Drive)
LAO_LO, LAO_HI = 0x0E80, 0x0EFF
def _has_lao(s): return any(LAO_LO <= ord(c) <= LAO_HI for c in s)

def learn_lao_tokens(lao_texts, cfg, prefix="lao_bpe"):
    with open("lao_corpus.txt","w",encoding="utf-8") as f:
        f.write("\n".join(t for t in lao_texts if t.strip()))
    spm.SentencePieceTrainer.train(
        input="lao_corpus.txt", model_prefix=prefix,
        vocab_size=cfg["lao_spm_vocab_size"], model_type="bpe",
        character_coverage=0.9995, hard_vocab_limit=False,
        unk_id=0, bos_id=-1, eos_id=-1, pad_id=-1)
    sp = spm.SentencePieceProcessor(model_file=prefix+".model")
    pieces = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]
    new, seen = [], set()
    for p in pieces:
        t = p.replace("▁","").strip()
        if t and _has_lao(t) and t not in seen:
            seen.add(t); new.append(t)
    print(f"SentencePiece: {len(pieces)} pieces -> {len(new)} Lao tokens.")
    return new

def learn_or_load_lao_tokens(lao_texts, cfg):
    os.makedirs(cfg["output_root"], exist_ok=True)
    cache = os.path.join(cfg["output_root"], "lao_new_tokens.json")
    if os.path.exists(cache):
        toks = json.load(open(cache, encoding="utf-8"))
        print(f"Loaded {len(toks)} cached Lao tokens from Drive.")
        return toks
    toks = learn_lao_tokens(lao_texts, cfg)
    json.dump(toks, open(cache,"w",encoding="utf-8"))
    print("Cached Lao tokens to Drive.")
    return toks

# MODEL + VOCAB EXTENSION
def load_extended(base_id, new_tokens, cfg):
    tok = AutoTokenizer.from_pretrained(base_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(base_id)
    existing = set(tok.get_vocab().keys())
    to_add = [t for t in new_tokens if t not in existing][: cfg["max_new_tokens_to_add"]]
    n_added = tok.add_tokens(to_add)
    if n_added > 0:
        model.resize_token_embeddings(len(tok))
        with torch.no_grad():
            emb = model.get_input_embeddings().weight
            emb[-n_added:] = emb[:-n_added].mean(dim=0).unsqueeze(0).repeat(n_added,1)
    print(f"{base_id}: +{n_added} tokens -> vocab {len(tok)}")
    return tok, model

# PREPROCESS + METRICS
def make_preprocess(tok, cfg, src_key, tgt_key, lang_token=None):
    def _fn(batch):
        srcs = batch[src_key]
        if lang_token:
            srcs = [f"{lang_token} {t}" for t in srcs]
        enc = tok(srcs, max_length=cfg["max_len"], truncation=True)
        lab = tok(text_target=batch[tgt_key], max_length=cfg["max_len"], truncation=True)
        enc["labels"] = lab["input_ids"]
        return enc
    return _fn

def _pick_bleu_tokenizer(tgt_lang):
    if tgt_lang != "lo": return "13a"
    try:
        sacrebleu.BLEU(tokenize="flores200"); return "flores200"
    except Exception:
        print("flores200 unavailable -> 'char' for Lao BLEU."); return "char"

def make_compute_metrics(tok, tgt_lang):
    bleu_tok = _pick_bleu_tokenizer(tgt_lang)
    def _compute(eval_pred):
        preds, labels = eval_pred
        if isinstance(preds, tuple): preds = preds[0]
        preds  = np.where(preds  != -100, preds,  tok.pad_token_id)
        labels = np.where(labels != -100, labels, tok.pad_token_id)
        hyp = [s.strip() for s in tok.batch_decode(preds,  skip_special_tokens=True)]
        ref = [s.strip() for s in tok.batch_decode(labels, skip_special_tokens=True)]
        refs = [ref]
        return {
            "bleu":   round(sacrebleu.corpus_bleu(hyp, refs, tokenize=bleu_tok).score, 2),
            "chrf":   round(sacrebleu.corpus_chrf(hyp, refs, word_order=0).score, 2),
            "chrf++": round(sacrebleu.corpus_chrf(hyp, refs, word_order=2).score, 2),
        }
    return _compute


def make_training_args(direction, cfg):
    out = os.path.join(cfg["output_root"], direction)
    kw = dict(
        output_dir=out,
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["grad_accum"],
        learning_rate=cfg["lr"], num_train_epochs=cfg["epochs"],
        weight_decay=0.01, warmup_ratio=0.05, lr_scheduler_type="cosine",
        label_smoothing_factor=cfg["label_smoothing"],
        predict_with_generate=True,
        generation_max_length=cfg["max_len"], generation_num_beams=cfg["num_beams"],
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        save_strategy="steps", save_steps=300, save_total_limit=2,
        load_best_model_at_end=False,
        report_to="none", seed=SEED,
    )
    params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"
    kw[eval_key] = "epoch"
    return Seq2SeqTrainingArguments(**kw)

def run_pipeline(direction, ds, new_tokens, cfg, force_retrain=False):
    assert direction in ("en2lo","lo2en")
    base = cfg["base_en2lo"] if direction=="en2lo" else cfg["base_lo2en"]
    src_key, tgt_key = ("en","lo") if direction=="en2lo" else ("lo","en")
    tgt_lang = tgt_key
    lang_token = cfg.get("tgt_lang_token") if direction=="en2lo" else None

    save_dir = os.path.join(cfg["output_root"], direction)
    done_marker = os.path.join(save_dir, "TRAINING_DONE.txt")
    finished = os.path.exists(done_marker) and not force_retrain

    if finished:
        print(f"[{direction}] finished model found -> skipping training.")
        tok = AutoTokenizer.from_pretrained(save_dir)
        model = AutoModelForSeq2SeqLM.from_pretrained(save_dir)
    else:
        tok, model = load_extended(base, new_tokens, cfg)

    pre = make_preprocess(tok, cfg, src_key, tgt_key, lang_token)
    tokenized = ds.map(pre, batched=True, remove_columns=ds["train"].column_names)
    collator = DataCollatorForSeq2Seq(tok, model=model)
    args = make_training_args(direction, cfg)
    common = dict(model=model, args=args,
                  train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
                  data_collator=collator, compute_metrics=make_compute_metrics(tok, tgt_lang))
    try:    trainer = Seq2SeqTrainer(**common, processing_class=tok)
    except TypeError: trainer = Seq2SeqTrainer(**common, tokenizer=tok)

    if not finished:
        last_ckpt = get_last_checkpoint(save_dir) if os.path.isdir(save_dir) else None
        if last_ckpt: print(f"[{direction}] resuming from {last_ckpt}")
        trainer.train(resume_from_checkpoint=last_ckpt)
        trainer.save_model(save_dir); tok.save_pretrained(save_dir)
        with open(done_marker,"w") as f: f.write("done")

    test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    print(f"[{direction}] TEST:", {k:v for k,v in test_metrics.items() if k.startswith("test_")})
    return test_metrics

#  MANUAL CHECK
_MODEL_CACHE = {}
def get_trained(direction, cfg):
    if direction not in _MODEL_CACHE:
        path = os.path.join(cfg["output_root"], direction)
        tok = AutoTokenizer.from_pretrained(path)
        model = AutoModelForSeq2SeqLM.from_pretrained(path).to(DEVICE).eval()
        _MODEL_CACHE[direction] = (tok, model)
    return _MODEL_CACHE[direction]

def translate(text, direction, cfg, num_beams=None, max_len=256):
    tok, model = get_trained(direction, cfg)
    src = f"{cfg['tgt_lang_token']} {text}" if (direction=="en2lo" and cfg.get("tgt_lang_token")) else text
    inputs = tok(src, return_tensors="pt", truncation=True, max_length=max_len).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, num_beams=num_beams or cfg["num_beams"], max_length=max_len)
    return tok.batch_decode(out, skip_special_tokens=True)[0].strip()

def manual_check(source, direction, cfg, reference=None, max_len=256):
    tgt_lang = "lo" if direction=="en2lo" else "en"
    hyp = translate(source, direction, cfg, max_len=max_len)
    print(f"[{direction}] SOURCE    : {source}")
    print(f"          PREDICTION: {hyp}")
    if reference:
        bt = _pick_bleu_tokenizer(tgt_lang)
        b = sacrebleu.sentence_bleu(hyp,[reference],tokenize=bt).score
        c = sacrebleu.sentence_chrf(hyp,[reference],word_order=0).score
        cpp = sacrebleu.sentence_chrf(hyp,[reference],word_order=2).score
        print(f"          REFERENCE : {reference}")
        print(f"          BLEU {b:.2f} | chrF {c:.2f} | chrF++ {cpp:.2f}")
    print("-"*60)
    return hyp

print("\nAll functions defined.")

Device: cuda | torch 2.11.0+cu128

All functions defined.


In [4]:
from google.colab import drive
drive.mount("/content/drive")

CONFIG = prepare_data(CONFIG)
ds = build_datasets(CONFIG)
NEW_LAO_TOKENS = learn_or_load_lao_tokens(ds["train"]["lo"], CONFIG)

Mounted at /content/drive
Found data on Drive -> no upload needed.
Loaded 160,007 raw pairs.
DatasetDict({
    train: Dataset({
        features: ['en', 'lo'],
        num_rows: 140000
    })
    validation: Dataset({
        features: ['en', 'lo'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['en', 'lo'],
        num_rows: 2000
    })
})
Loaded 3241 cached Lao tokens from Drive.


In [6]:
metrics_en2lo = run_pipeline("en2lo", ds, NEW_LAO_TOKENS, CONFIG,force_retrain=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/707k [00:00<?, ?B/s]

...

[en2lo] resuming from /content/drive/MyDrive/nmt/runs/en2lo/checkpoint-4750


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss,Bleu,Chrf,Chrf++
2,4.006700,3.958606,19.420000,31.590000,29.630000
3,3.825200,3.891704,20.470000,32.690000,30.660000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[64109]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


[en2lo] TEST: {'test_loss': 3.877152442932129, 'test_bleu': 19.52, 'test_chrf': 31.98, 'test_chrf++': 29.83, 'test_runtime': 236.387, 'test_samples_per_second': 8.461, 'test_steps_per_second': 0.529}


In [5]:
metrics_lo2en = run_pipeline("lo2en", ds, NEW_LAO_TOKENS, CONFIG, force_retrain=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/1.21M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/810k [00:00<?, ?B/s]

...

[lo2en] resuming from /content/drive/MyDrive/nmt/runs/lo2en/checkpoint-4750


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss,Bleu,Chrf,Chrf++
2,4.221700,4.187951,16.910000,31.070000,28.820000
3,4.052500,4.137867,17.310000,31.350000,29.120000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[62306]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


[lo2en] TEST: {'test_loss': 4.084959983825684, 'test_bleu': 18.48, 'test_chrf': 32.29, 'test_chrf++': 30.12, 'test_runtime': 232.1076, 'test_samples_per_second': 8.617, 'test_steps_per_second': 0.539}


In [7]:
baseline = {"en2lo": 23.33, "lo2en": 29.41}
rows = [("EN->LO","en2lo",metrics_en2lo), ("LO->EN","lo2en",metrics_lo2en)]
print(f"{'Direction':<10}{'BLEU(base)':>12}{'BLEU(new)':>11}{'Delta':>8}{'chrF':>8}{'chrF++':>9}")
print("-"*58)
for name,key,m in rows:
    b=m.get("test_bleu"); c=m.get("test_chrf"); cpp=m.get("test_chrf++")
    print(f"{name:<10}{baseline[key]:>12.2f}{b:>11.2f}{b-baseline[key]:>+8.2f}{c:>8.2f}{cpp:>9.2f}")

NameError: name 'metrics_en2lo' is not defined

In [8]:

en2lo_sentences = [
    "Hello, how are you today?",
    "Good morning everyone, today I will talk about machine translation.",
    "Thank you very much for your help.",
    "I would like to visit Laos next year.",
    "The weather is very nice this morning.",
    "Please send me the report by tomorrow.",
    "Education is very important for our country.",
    "Can you tell me the way to the train station?",
]
for s in en2lo_sentences:
    manual_check(s, "en2lo", CONFIG)


lo2en_sentences = [
    "ສະບາຍດີ, ມື້ນີ້ເຈົ້າສະບາຍດີບໍ່?",
    "ຂອບໃຈຫຼາຍສຳລັບການຊ່ວຍເຫຼືອ.",
    "ຂ້ອຍຮັກປະເທດລາວ.",
    "ມື້ນີ້ອາກາດດີຫຼາຍ.",
    "ການສຶກສາມີຄວາມສຳຄັນຫຼາຍ.",
    "ສະບາຍດີທຸກຄົນ ມື້ນີ້ຂ້ອຍຢາກເວົ້າກ່ຽວກັບການແປພາສາ.",
]
for s in lo2en_sentences:
    manual_check(s, "lo2en", CONFIG)


manual_check("Thank you very much for your help today.", "en2lo", CONFIG,
             reference="ຂອບໃຈຫຼາຍສຳລັບການຊ່ວຍເຫຼືອຂອງເຈົ້າມື້ນີ້.")
manual_check("ຂ້ອຍຮັກເຈົ້າ", "lo2en", CONFIG, reference="I love you")



[en2lo] SOURCE    : Hello, how are you today?
          PREDICTION: ສະບາຍດີ , ທ່າ ນ ໃນມື້ນີ້ ແນວໃດ ?
------------------------------------------------------------
[en2lo] SOURCE    : Good morning everyone, today I will talk about machine translation.
          PREDICTION: ວັນ ສຸກ ທຸກຄົນ , ໃນມື້ນີ້ ຂ້າພະເຈົ້າ ຈະ ເວົ້າ ກ່ຽວກັບ ກາ ນ transposh machine.
------------------------------------------------------------
[en2lo] SOURCE    : Thank you very much for your help.
          PREDICTION: ຂໍຂອບໃຈ ຫຼາຍ ສໍາລັບການ ຊ່ວຍເຫຼືອ ຂອງທ່ານ .
------------------------------------------------------------
[en2lo] SOURCE    : I would like to visit Laos next year.
          PREDICTION: ຂ້າພະເຈົ້າ ຕ້ອງການທີ່ຈະ ໄປຢ້ຽມຢາມ Laos ປີ ຕໍ່ໄປ .
------------------------------------------------------------
[en2lo] SOURCE    : The weather is very nice this morning.
          PREDICTION: ສະພາບອາກາດໃນ ซะโมิซละนดซ - ການຄາດຄະເນ ດິນຟ້າອາກາດ ສໍາລັບມື້ນີ້ ມື້ອື່ນ , ອາທິດ
---------------------------------------------------------

'I love you'

In [1]:
import shutil
from google.colab import files

# zip and download the final EN-LO model
shutil.make_archive("/content/en2lo_model", "zip", "/content/drive/MyDrive/nmt/runs/en2lo")
files.download("/content/en2lo_model.zip")

# zip and download the final LO-EN model
shutil.make_archive("/content/lo2en_model", "zip", "/content/drive/MyDrive/nmt/runs/lo2en")
files.download("/content/lo2en_model.zip")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/nmt/runs/en2lo'